# 【学習用・解説付き写し】S6E8 Rank-Logit-Regime Fusion | LB0.97125- **コンペ**: [Predicting Smartphone Addiction（Playground Series S6E8）](https://www.kaggle.com/competitions/playground-series-s6e8)- **原著者**: BYER (hboyang)- **元notebook**: https://www.kaggle.com/code/hboyang/s6e8-rank-logit-regime-fusion-lb0-97125- **Public Score**: 0.97125（Version 3 / 実行時間 10分42秒・GPU T4 x2）— 本日時点でこのコンペの**公開notebook最高スコア**- **コンペ終了まで**: 残り7日## 手法の概要（1段落）自分でモデルを1から学習するのではなく、**公開されている6人分のOOF（out-of-fold）予測ライブラリを集めて205メンバーのプールを作り、それを2段階でスタッキングする**notebookです。各メンバーの予測を(a) **順位に変換したもの（rank01）** と (b) **ロジット変換したもの（logit）** の両方を特徴量として横に並べ、float64のロジスティック回帰（LBFGS）で融合します（= "dual" ストリーム）。さらに、**元データの欠損数**（欠損ゼロ / 4個以上欠損）とメンバー間の**意見のばらつき（disagreement）**を交互作用項として掛け合わせた「レジーム特徴量」でもう1本ロジスティック回帰を学習し（= "regime" ストリーム）、最後に両者を **順位空間で 0.55 : 0.45 に混ぜて**提出しています。> 📌 これは**学習目的の解説付き写し**です。原著者のコードは変更していません（出力を削除し、> 読みやすさのために1つの巨大なコードセルを機能ごとに分割し、各ブロックの前に日本語解説を挿入しただけです）。

## 評価指標（簡潔に）**タスク**: 各ユーザーが「スマホ依存（`addicted_label`）」かどうかを当てる**二値分類**。train 691,369行 / test 296,302行の合成表形式データです。**指標**: **ROC-AUC**（Area Under the ROC Curve）。「ランダムに選んだ陽性1件と陰性1件を並べたとき、陽性のほうに高いスコアを付けられる確率」に等しい指標で、**予測値の絶対的な大きさではなく順序（ランキング）だけを見ます**。0.5がランダム、1.0が完璧。**なぜこの手法設計が指標に噛み合っているか**:- AUCが順序しか見ないので、このnotebookは全メンバーの予測を最初に **`rank01`（順位を0〜1に正規化）** へ変換します。  こうすると「あるモデルは0.2〜0.3に予測が集中、別のモデルは0〜1に散らばる」といった**スケールの違いが消え**、  素直に平均できるようになります。**AUC指標のコンペでrank averageが定番なのはこのため**です。- 一方でrank変換は「どれくらい自信があるか」の情報を捨てます。そこで **`logit`（オッズの対数）** 版も同時に並べ、  順序情報と確信度情報の両方を線形モデルに渡しています。これが "Rank-Logit" の意味です。- LB 0.9712 付近では**上位陣の差は0.0001程度**。この帯域では「強いモデルを1つ足す」より  「**誤りの相関が低いメンバーを混ぜる**」ほうが効くため、多様なOOFを大量に集める戦略が合理的になります。> ⚠️ ただし後半の「改善点」でも触れますが、**公開OOFを大量に混ぜる手法はPublic LBに過剰適合しやすい**という> 批判が同コンペのDiscussion/公開notebookで繰り返し出ています（例: "Why Every S6E8 Notebook Above 0.97110 Overfits"）。> スコアの高さをそのまま手法の良さと読み替えないこと。

# S6E8 Rank-Logit-Regime Fusion | LB0.97125

## Inputs

Attach the competition data, the aligned public OOF libraries, the CC0 weak-50 library, and `hboyang/s6e8-catstrall-member`. The code uses positional arrays in the original train/test row order.

The public libraries are from najiama, boltuzamaki, szymonkapiski, dariushafshar, raykkretzschmar, and adarsh1077. The weak-50 library is from szymonkapiski.


## Method and acknowledgements

The pool combines diverse fold-safe OOF predictions, adds one quantized CatBoost member and several aligned diversity members, then fits a rank+logit logistic stack plus a missingness/disagreement regime stack. Their rank blend is combined with a cross-fitted base stack using a fixed alpha selected by honest validation.

Thanks to the authors of the attached public OOF libraries for making their predictions available. This notebook only reuses their released prediction files and keeps the fusion code explicit.


### ブロック1: 冒頭のdocstringとimport（What / Why）**何をしているか**: このスクリプトが何をするものかをdocstringで宣言し、必要なライブラリを読み込みます。**なぜここが重要か**: このdocstringは単なる説明ではなく、**再現のための契約書**になっています。特に:```StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(train, train.addicted_label)over train.csv in ORIGINAL FILE ROW ORDER.All member OOF/test arrays are POSITIONAL (no ids)```これは「**コミュニティ全体で同じfold分割を凍結して使っている**」という宣言です。スタッキングでは、各メンバーのOOF予測が**まったく同じfold分割**から作られていないと、あるメンバーだけが「自分の検証データを学習に使ってしまった」状態になり、スタッキング層がそのメンバーを不当に高く評価します（**外来foldによるOOF楽観バイアス**）。また `POSITIONAL (no ids)` は「配列の i 番目が train.csv の i 行目に対応する。IDでの結合はしない」という意味。IDで結合しないぶん高速ですが、**1行でもズレたら全メンバーが静かに壊れます**。だからこそ行数（691,369 / 296,302）をdocstringに明記しているわけです。自分でスタッキングを組むときも、この2点（fold定義と行の対応規則）は必ずコードの先頭に書き残してください。> 用語: **OOF（out-of-fold）予測** = 交差検証で、各サンプルが「学習に使われなかった fold」にいるときに得られた予測。> スタッキングの2段目はこれを入力にすることで、リークを避けつつ全学習データ分の入力を確保します。

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Open S6E8 205-member fusion.

Reconstructs the full "varB" pool from public OOF libraries + a small set of
locally-trained members, refits the current-meta-stack base member on the fly,
and runs the rank+logit+regime GPU fusion. Produces submission.csv and prints
live out-of-fold ROC-AUC.

Fold alignment (community frozen):
    StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(train, train.addicted_label)
over train.csv in ORIGINAL FILE ROW ORDER. All member OOF/test arrays are
POSITIONAL (no ids): OOF = train.csv order (691,369), test = test.csv order
(296,302).

Inputs (override with env vars; defaults target a Kaggle notebook mount):
    DATA_ROOT   default /kaggle/input/playground-series-s6e8
    NAJI_ROOT   default /kaggle/input/predicting-smartphone-addiction-oof-submission-csv
    BOLT_ROOT   default /kaggle/input/s6e8-oof-prediction-library
    SZYMON_ROOT default /kaggle/input/s6e8-oof-library-47-models
    FM_ROOT     default /kaggle/input/s6e8-fm-lattice-blend-members
    GOLEM_ROOT  default /kaggle/input/s6e8-golem-oof-library
    ADARSH_ROOT default /kaggle/input/s6e8-adarsh-oof-library
    OUR_ROOT    default /kaggle/input/s6e8-local-fusion-members
    OUT_DIR     default /kaggle/working
"""
from __future__ import annotations

import datetime
import gc
import glob
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn.functional as F



### ブロック2: メンバー名のリスト（What / Why）**何をしているか**: 融合プールに入れる各予測モデル（メンバー）の名前を並べたリストです。**なぜそうするのか**: 名前を**明示的にリストで固定**しておくと、「ディレクトリにある `.npy` を全部読む」方式で起きがちな事故——データセットが更新されてファイルが増減し、**昨日と今日でプールの中身が変わってスコアが再現しない**——を防げます。また、名前が並んでいること自体が**プロヴェナンス（出所）の記録**になります。どのメンバーが誰の公開ライブラリ由来で、どれが自前の学習かが名前から追える設計です。スタッキングでは「同じ人の同じ設定違いモデルが50個」といった**見かけ上の多様性**が混ざりやすいので、出所を追える状態にしておくことは精度以前に**解釈のために**必要です。

In [ ]:
# ---------------------------------------------------------------------------
# Member registry: 145 names from the community pools + 4 local extras.
# naji*   -> najiama blends
# bolt_*  -> boltuzamaki library
# sz_*    -> szymonkapiski library
# fm_*    -> raykkretzschmar fm-lattice
# golem_* -> dariushafshar golem library
# a_*     -> adarsh1077 library
# candidate_*/fresh_*/local_* -> OUR locally trained members (our dataset)
# ---------------------------------------------------------------------------
MEMBER_NAMES = [
 "naji07","naji08","naji09","naji10","naji12","naji13","naji14","naji16",
 "candidate_naji16_boltwide185_lookupv2_165_xgb10_01_rank","candidate_naji16_boltwide1877_lookup1649_x0047_fm003_rank",
 "candidate_naji16_bolt_lookupv3v2_coord_rank","candidate_naji16_bolt_deepfm_coord_rank",
 "candidate_naji16_bolt_final_coord_rank","candidate_naji16_bolt_lookup256l8_rank",
 "bolt_xgb_hpo_d7","bolt_xgb_te_5fold","bolt_xgb_te_4fold","bolt_xgb_d7_alt1","bolt_xgb_d7_alt2",
 "bolt_xgb_dd_d4","bolt_xgb_dd_d5","bolt_xgb_dd_d6","bolt_cat_nested_te","bolt_cat_dual_view",
 "bolt_cat_dual_seed81","bolt_cat_cpu5","bolt_cat_pair_evidence","bolt_lgb_te_5fold","bolt_lgb_pair_lattice",
 "bolt_lgb_driver_recon","bolt_lgb_missing_global","bolt_lgb_raw_d6","bolt_histgb_5fold","bolt_xgb_raw_bag",
 "bolt_repr_lgb_global","bolt_lookup_v1","bolt_lookup_v2_s03","bolt_lookup_v2_s81","bolt_lookup_v2_s1037",
 "bolt_lookup_v2_s42","bolt_lookup_v2_s959","bolt_lookup_v3_evidence","bolt_realmlp_lattice","bolt_deepfm_exact",
 "bolt_fttransformer","bolt_dcnv2_cross","bolt_gandalf_gflu","bolt_tabr_retrieval","bolt_ebm_exact",
 "bolt_foldsafe_te_xgb","bolt_foldsafe_te_xgb_10f","bolt_foldsafe_te_cat","bolt_foldsafe_te_multi",
 "bolt_foldsafe_te_wide","bolt_lookup_v2_s20260901",
 "sz_altview","sz_cat","sz_cat_tuned","sz_digit_cat","sz_digit_lgbm","sz_digit_xgb","sz_hgb",
 "sz_imp_cat","sz_imp_lgbm","sz_imp_lgbm_tuned","sz_imp_xgb","sz_imp_xgb_tuned","sz_lat_cat","sz_lat_lgbm",
 "sz_lat_lgbm_s5","sz_lat_xgb","sz_latmax_lgbm","sz_latr1_lgbm","sz_latr1_xgb","sz_lattri_lgbm","sz_lattri_xgb",
 "sz_latwide_cat","sz_latwide_lgbm","sz_latwide_xgb","sz_lgbm","sz_lgbm_tuned","sz_lookup",
 "sz_naji01","sz_naji02","sz_naji03","sz_naji04","sz_naji05","sz_pub_cat","sz_pub_donlgbm","sz_pub_evg",
 "sz_pub_ravi","sz_pub_resnet","sz_pub_rmlp","sz_pub_ryota","sz_pub_tabm","sz_pub_tabnet","sz_pubfe_cat",
 "sz_pubfe_lgb","sz_pubfe_xgb","sz_pubmk_cat","sz_pubmk_nn","sz_rmlp_lat","sz_rmlp_lat3","sz_tabm_bounds",
 "sz_tabm_deep","sz_tabm_deeper","sz_tabm_div","sz_tabm_imp","sz_tabm_seed3","sz_tabm_wide","sz_tabm_x12",
 "sz_view_bounds_cat","sz_view_bounds_lgbm","sz_view_nolattice_lgbm","sz_view_rank_cat","sz_view_rank_lgbm",
 "sz_view_resid_lgbm","sz_xgb","sz_xgb_tuned",
 "fm_fmdeep","fm_fmnum","fm_fmplr","fm_fmpure","fm_fmwide",
 "golem_a","golem_b","golem_c","golem_d","golem_e","golem_f","golem_g",
 "fresh_tabm_fresh_rich_s2026","fresh_realmlp_fresh_s2026","fresh_lookup_fresh_d256_l8_s5150",
 "fresh_lookup_fresh_d384_l6_s2718","fresh_cat_fresh_d9_s606","fresh_xgb_fresh_d6_s606","fresh_xgb_fresh_d7_s314159",
 "naji18","naji19",
 "a_logregte","a_catnative","a_gxgbnote","a_glgbnote2","a_gcatnote",
 "local_tabm_rich_seed3","local_tabm_rich_alt","local_tabm_rich_seed909","local_lookup_d384_l4",
]




### ブロック3: 設定値（環境変数・ハイパーパラメータ）（What / Why）**何をしているか**: 入力データの場所を環境変数から読み（`_env`）、デバイス（GPU/CPU）・fold数(5)・seed(42)、そして融合のハイパーパラメータ `FUSION_C = 3.5`（ロジスティック回帰の正則化の逆数）、`MIX_W = 0.55`（2ストリームの混合比）を定義します。**なぜそうするのか**:- `_env(name, default)` で環境変数から上書きできるようにしておくと、**Kaggle上でもローカルでも同じコードが動きます**。  パスのハードコードは再現性の敵です。- `FUSION_C = 3.5` は **L2正則化の強さの逆数**。Cが大きいほど正則化が弱い＝各メンバーの重みが自由に動きます。  205メンバーは互いに強く相関している（多重共線性）ので、正則化が弱すぎると  「+8.2 と -7.9 が打ち消し合う」ような不安定な係数が生まれ、testでわずかにデータが変わると崩れます。  逆に強すぎると全員ほぼ均等になり、rank averageと変わらなくなる。**3.5はその中間を実測で選んだ値**です。- `SEED = 42` と `N_FOLD = 5` は**コミュニティ凍結値**なので、ここを変えると持ち込んだOOFと整合しなくなります。  自分の実験だから、と安易に変えてはいけない箇所です。

In [ ]:
def _env(name, default):
    return Path(os.environ.get(name, default))


DATA_ROOT = _env("DATA_ROOT", "/kaggle/input/competitions/playground-series-s6e8")
WEAK50_ROOT = _env("WEAK50_ROOT", "/kaggle/input/datasets/szymonkapiski/s6e8-50-weakest-oof-models")
EXTRA_ROOT = _env("EXTRA_ROOT", "/kaggle/input/datasets/hboyang/s6e8-catstrall-member")
NAJI_ROOT = _env("NAJI_ROOT", "/kaggle/input/datasets/najiama/predicting-smartphone-addiction-oof-submission-csv")
BOLT_ROOT = _env("BOLT_ROOT", "/kaggle/input/datasets/boltuzamaki/s6e8-oof-prediction-library")
SZYMON_ROOT = _env("SZYMON_ROOT", "/kaggle/input/datasets/szymonkapiski/s6e8-oof-library-47-models")
FM_ROOT = _env("FM_ROOT", "/kaggle/input/datasets/raykkretzschmar/s6e8-fm-lattice-blend-members")
GOLEM_ROOT = _env("GOLEM_ROOT", "/kaggle/input/datasets/dariushafshar/s6e8-golem-oof-library")
ADARSH_ROOT = _env("ADARSH_ROOT", "/kaggle/input/datasets/adarsh1077/s6e8-adarsh-oof-library")
OUR_ROOT = _env("OUR_ROOT", "/kaggle/input/datasets/hboyang/s6e8-150-fusion-local-members")
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
N_FOLD = 5
SEED = 42
# Validation switch: False = skip the 5-fold OOF / per-fold alpha (fast, runs
# only the required meta-stack refit + full-data fits; test predictions are the
# same either way). True = also recompute the honest 5-fold OOF and nested alpha.
VALIDATE = False
FUSION_C = 3.5
MIX_W = 0.55
ALPHA_DEFAULT = 0.7




### ブロック4: 小さな変換ユーティリティ（What / Why）**何をしているか**: 4つの短い関数を定義します。- `rank01(v)`: `np.argsort` を2回かけて**順位**を求め、`(rank + 0.5) / n` で 0〜1 に正規化します。- `logit(v)`: 確率 p を `log(p / (1-p))` に変換します。`np.clip(v, 1e-6, 1-1e-6)` で0除算・無限大を防いでいます。- `find_files` / `find_test_near`: OOFファイルに対応するtestファイルを、同じディレクトリ→配下全体の順に探します。**なぜそうするのか**:- **`np.argsort` の二重適用**は「順位を得る」定番イディオムです。1回目で「小さい順に並べたときの元インデックス」、  2回目でその逆写像＝各要素の順位が得られます。`scipy.stats.rankdata` より速いので大規模データで好まれます。- **`+0.5` を足す理由**: そのまま `rank / n` にすると最小値が 0、最大値が (n-1)/n になり、端が非対称になります。  0.5を足すと 0 と 1 のちょうど内側に対称に収まり、この後 `logit` を通しても無限大になりません。地味ですが重要な処理です。- **`logit` を使う理由**: 確率のまま平均すると、0.99 と 0.999（オッズで10倍違う）の差がほとんど潰れます。  ロジット空間なら「確信度の差」が線形に効くので、線形モデルの入力として素直になります。- **ファイル探索を関数化する理由**: 6つの公開ライブラリはそれぞれディレクトリ構成が違います。  「まず隣を見て、無ければ再帰的に探す」というフォールバックを1か所にまとめておくと、  新しいライブラリを追加するときに触る場所が増えません。

In [ ]:
def rank01(v):
    return (np.argsort(np.argsort(v)) + 0.5) / v.size


def logit(v):
    v = np.clip(v, 1e-6, 1.0 - 1e-6)
    return np.log(v / (1.0 - v))


def find_files(root: Path, pattern: str) -> list[Path]:
    hits = list(root.glob(pattern))
    if not hits:
        hits = list(root.glob("**/" + pattern))
    return sorted(hits)


def find_test_near(oof_file: Path, name: str, root: Path) -> Path | None:
    """Locate test_<name>.npy: same dir first, else anywhere under root."""
    cand = oof_file.with_name(f"test_{name}.npy")
    if cand.exists():
        return cand
    hits = find_files(root, f"test_{name}.npy")
    return hits[0] if hits else None




### ブロック5: `build_pool()` — 205メンバーのプール構築（What / Why）**何をしているか**: train/test CSVを読み、6人分の公開OOFライブラリ＋自前メンバー＋弱いモデル50個（weak50）をそれぞれ `.npy` から読み込み、**全部を `rank01` で順位化してから横に連結**して、`oof`（691,369行 × メンバー数）と `test`（296,302行 × メンバー数）の2つの行列を返します。**なぜそうするのか**:- **なぜ「弱いモデル50個」をわざわざ入れるのか**: スタッキングで効くのは個々の強さではなく  **誤りの非相関性**です。単体AUCが0.960しかないモデルでも、上位モデルが揃って間違えるサンプルで  正解していれば、線形結合で全体を押し上げます。逆に単体0.9710のモデルを10個足しても、  それらが同じデータの同じ癖を学んでいれば情報はほとんど増えません。- **なぜ読み込み時点でrank化するのか**: メンバーごとに出力のスケールが違う（sigmoid出力・生のスコア・  すでにrank済み、など）ため、統一しないと連結した行列の列同士が比較不能になります。  評価指標がAUC（順序のみ）なので、rank化による情報損失は最小です。- **注意すべき罠**: ここで読み込む公開OOFが**本当にfold凍結を守っているか**は、読み込む側からは検証できません。  実務では、読み込んだ各メンバーの単体OOF AUCを表示し、**公称値と一致するか**を確認する検証を入れるべきです  （一致しなければfoldが違う＝そのメンバーは信用できない）。

In [ ]:
def build_pool():
    """Return aligned (ranked) OOF matrix, test matrix, member name list."""
    train = pd.read_csv(DATA_ROOT / "train.csv")
    test = pd.read_csv(DATA_ROOT / "test.csv")
    naji_r = NAJI_ROOT
    bolt_r = BOLT_ROOT
    szymon_r = SZYMON_ROOT
    fm_r = FM_ROOT
    golem_r = GOLEM_ROOT
    adarsh_r = ADARSH_ROOT
    ours_r = OUR_ROOT
    n, nt = len(train), len(test)
    y = train.addicted_label.to_numpy(np.int8)
    extra_names = [f"weak50_m{j + 1:02d}" for j in range(50)] + [
        "kirill_o1", "koda_exact_te", "stringify_str3_d6",
        "stringify_strall_d6", "stringify_str3derived_d7", "cat_strall_d8",
    ]
    wanted = set(MEMBER_NAMES) | set(extra_names)
    store: dict[str, tuple[np.ndarray, np.ndarray]] = {}

    def add(name, oof, tst):
        if name not in wanted:
            return
        oof = np.asarray(oof).reshape(-1)
        tst = np.asarray(tst).reshape(-1)
        if len(oof) == n and len(tst) == nt and np.isfinite(oof).all() and np.isfinite(tst).all():
            store[name] = (rankdata(oof).astype(np.float32) / n,
                           rankdata(tst).astype(np.float32) / nt)

    # najiama blends
    for path in find_files(naji_r, "*_blend_oof_predictions.csv"):
        p = path.name[:2]
        q = path.with_name(f"{p}_blend_submission.csv")
        if not q.exists():
            q = path.with_name(f"{p}_blend_submission.csv.csv")
        if q.exists():
            add(f"naji{p}", pd.read_csv(path, usecols=["addicted_label"]).addicted_label,
                pd.read_csv(q, usecols=["addicted_label"]).addicted_label)

    # boltuzamaki parquet
    for pq in find_files(bolt_r, "oof_predictions.parquet")[:1]:
        tq = pq.with_name("test_predictions.parquet")
        oof = pd.read_parquet(pq)
        tst = pd.read_parquet(tq)
        for c in oof.columns:
            if c != "id" and c in tst:
                add(f"bolt_{c}", oof[c], tst[c])

    # szymonkapiski npy (oof_<m>.npy + test_<m>.npy)
    for path in find_files(szymon_r, "oof_*.npy"):
        m = path.stem[4:]
        q = find_test_near(path, m, SZYMON_ROOT)
        if q is not None:
            add(f"sz_{m}", np.load(path), np.load(q))

    # fm-lattice + golem (same convention)
    for root, prefix in ((fm_r, "fm_"), (golem_r, "golem_")):
        for path in find_files(root, "oof_*.npy"):
            m = path.stem[4:]
            q = find_test_near(path, m, root)
            if q is not None:
                add(prefix + m, np.load(path), np.load(q))

    # adarsh library
    for path in find_files(adarsh_r, "oof_*.npy"):
        m = path.stem[4:]
        q = find_test_near(path, m, ADARSH_ROOT)
        if q is not None:
            add(f"a_{m}", np.load(path), np.load(q))

    # OUR locally-trained members (flat: oof_<name>.npy / test_<name>.npy)
    for path in find_files(ours_r, "oof_*.npy"):
        m = path.stem[4:]
        q = find_test_near(path, m, OUR_ROOT)
        if q is not None:
            add(m, np.load(path), np.load(q))

    # CC0 weak-50 diversity library.
    weak_oof = np.load(WEAK50_ROOT / "oof.npy", mmap_mode="r")
    weak_test = np.load(WEAK50_ROOT / "test.npy", mmap_mode="r")
    if weak_oof.shape != (n, 50) or weak_test.shape != (nt, 50):
        raise RuntimeError(f"Unexpected weak-50 shapes: {weak_oof.shape} / {weak_test.shape}")
    for j, name in enumerate(extra_names[:50]):
        add(name, weak_oof[:, j], weak_test[:, j])

    # Six aligned incremental members from the companion public dataset.
    for name in extra_names[50:]:
        add(name, np.load(EXTRA_ROOT / f"oof_{name}.npy"),
            np.load(EXTRA_ROOT / f"test_{name}.npy"))

    missing = [name for name in MEMBER_NAMES if name not in store]
    if missing:
        raise RuntimeError(
            f"Missing {len(missing)} members (check attached inputs / layouts): {missing}")
    names = MEMBER_NAMES + extra_names
    oof = np.column_stack([store[nm][0] for nm in names])
    tst = np.column_stack([store[nm][1] for nm in names])
    return oof, tst, names, y, train, test




### ブロック6: `regime_feats()` — 「状況（レジーム）」を表す特徴量（What / Why）**何をしているか**: 融合特徴量 `b`（rankとlogitを連結したもの）に対して、次を掛け合わせた行列を作ります。- `complete`: そのユーザーの元データに**欠損がゼロ**なら1- `missing_many`: **欠損が4個以上**なら1- `d`: メンバー間予測の**標準偏差を標準化したもの**＝「モデル同士がどれくらい割れているか」- 集約統計: メンバー予測の平均・標準偏差・レンジそして `b`, `b × complete`, `b × missing_many`, `b × d`, 集約統計 を横に連結して返します。**なぜそうするのか**: ここが本notebookの一番のオリジナル部分です。通常のスタッキングは「メンバーAの重みは全サンプルで一定」と仮定します。しかし現実には、**欠損だらけのサンプルでは木モデルが強く、欠損の無いサンプルでは線形モデルが強い**、といった状況依存が起きます。`b × complete` のような**交互作用項**を入れると、線形モデルのまま「欠損ゼロのときはメンバーAを重く、欠損が多いときはメンバーBを重く」という**条件付きの重み**を学習できます。`b × d`（意見の割れ具合との交互作用）はさらに面白く、「**モデル同士が割れているサンプルでだけ、別の重み付けに切り替える**」ことを意味します。簡単なサンプルは誰が予測しても当たるので差がつかず、勝負は難しいサンプルで決まる——という直観をモデル化しています。> 用語: **交互作用項（interaction term）** = 2つの特徴量の積。線形モデルに「AがONのときだけBの効き方が変わる」> という非線形性を持ち込める、最も安価な手段です。> 副作用として**列数が数倍に膨れる**ので、正則化とメモリ管理がセットで必要になります。

In [ ]:
def regime_feats(r_half, b, complete, missing_many):
    b = np.asarray(b)
    complete = np.asarray(complete)
    missing_many = np.asarray(missing_many)
    d = r_half.std(axis=1)
    d = (d - d.mean()) / (d.std() + 1e-12)
    agg = np.column_stack([
        r_half.mean(1), r_half.std(1), r_half.max(1) - r_half.min(1),
        complete, missing_many,
    ])
    return np.column_stack([b, b * complete[:, None], b * missing_many[:, None], b * d[:, None], agg])




### ブロック7: `fit_logreg()` — float64 + LBFGS + チャンク勾配のロジスティック回帰（What / Why）**何をしているか**: scikit-learnではなく、PyTorchで自作したロジスティック回帰です。特徴は3点。1. **float64（倍精度）** で計算する2. **LBFGS**（強Wolfe条件のline search付き）で最適化する3. 勾配を **262,144行ずつのチャンクに分けて足し合わせる****なぜそうするのか**:- **なぜfloat64か**: 融合行列は約 1205列 × 691,369行。メンバー同士が強く相関しているため、  ヘッセ行列の条件数が非常に悪くなります。float32だと丸め誤差で係数が振動し、  実行のたびに結果が変わってしまいます。AUCの差が0.0001を争う世界では致命的です。- **なぜLBFGSか**: SGD/Adamのような確率的手法は学習率とエポック数の調整が要り、収束点も揺れます。  LBFGSは**準ニュートン法**で、二次の曲率情報を近似的に使うため、凸問題（ロジスティック回帰は凸）では  少ない反復で高精度に収束します。`tolerance_grad=1e-10` という厳しい停止条件を置けるのもこのためです。- **なぜチャンクに分けるか**: 1205 × 691,369 のfloat64行列は約6.6GB。勾配計算の中間結果も含めると  Kaggle T4（16GB）に載りません。**数学的にはまったく同じ全バッチ勾配**を、  部分和に分けて累積することで**メモリ上限を回避**しています。  docstringが「差は float64 の加算順序に由来する ~1e-14 の相対誤差だけ」と明記しているのが誠実です。**ここから学べる一般則**: 「メモリが足りない」ときの選択肢は(a) 精度を落とす（float32化）、(b) データを減らす（サブサンプリング）、(c) **計算を分割する**、の3つ。(a)(b)は結果を変えますが、(c)だけは**結果を変えずに**制約を外せます。まず(c)を検討する癖をつけるとよいです。

In [ ]:
def fit_logreg(x_train, y_train, x_valid, c=FUSION_C, chunk=262144):
    """Full-batch (64-bit) logistic fit with chunked gradient accumulation.

    Mathematically the same float64 LBFGS as the A800-validated version; only the
    gradient is summed in chunks so the full regime matrix (~1205 x 691k float64)
    never blows past ~16 GB on a Kaggle T4. Difference in results: float64
    summation-order noise only (~1e-14 relative), far below platform noise.
    """
    x_train = np.asarray(x_train, dtype=np.float64)
    y_train = np.asarray(y_train, dtype=np.float64)
    x_valid = np.asarray(x_valid, dtype=np.float64)
    xt = torch.as_tensor(x_train, device=DEVICE, dtype=torch.float64)
    yt = torch.as_tensor(y_train, device=DEVICE, dtype=torch.float64)
    xv = torch.as_tensor(x_valid, device=DEVICE, dtype=torch.float64)
    n = xt.shape[0]
    model = torch.nn.Linear(xt.shape[1], 1, bias=True, device=DEVICE, dtype=torch.float64)
    torch.nn.init.zeros_(model.weight)
    torch.nn.init.zeros_(model.bias)
    opt = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=1000, max_eval=1500,
                            tolerance_grad=1e-10, tolerance_change=1e-15, history_size=50,
                            line_search_fn="strong_wolfe")
    reg = 1.0 / (2.0 * c * n)

    def closure():
        opt.zero_grad(set_to_none=True)
        total = None
        for i in range(0, n, chunk):
            xb = xt[i:i + chunk]
            yb = yt[i:i + chunk]
            logits = model(xb).squeeze(1)
            cl = F.binary_cross_entropy_with_logits(logits, yb, reduction="sum") / n
            cl.backward()
            total = cl.detach() if total is None else total + cl.detach()
        return total + reg * model.weight.square().sum()

    opt.step(closure)
    with torch.no_grad():
        valid = model(xv).squeeze(1).cpu().numpy()
    del xt, yt, xv, model, opt
    torch.cuda.empty_cache()
    return valid



### ブロック8: `main()` — 全体の流れ（What / Why）**何をしているか**: 実行の本体です。順に:1. `build_pool()` でOOF/test行列を構築2. 元データの**欠損個数**から `complete` / `missing_many` フラグを作る3. `StratifiedKFold(5, shuffle=True, random_state=42)` でfoldを再現4. **メタスタックのベースメンバーをその場で再学習**（`StandardScaler` + `LogisticRegression(C=0.1)` を5-fold CVで）し、   得られたOOF/test予測を**融合行列の追加列として加える**5. 全列を `rank01` と `logit` に変換して横に連結（`bf`, `bt`）→ `fit_logreg` で **dualストリーム**を学習6. `regime_feats` でレジーム行列を作り、平均0・分散1に標準化 → `fit_logreg` で **regimeストリーム**を学習7. 両者の予測を**順位空間で 0.55 : 0.45 に混ぜて**最終予測にし、`submission.csv` を書き出す**なぜそうするのか / 注目点**:- **なぜベースメンバーを「その場で再学習」するのか**: これは`C=0.1`（強い正則化）の素直なメタスタックです。  弱正則化のdualストリームと**別の性格の予測**になるので、それ自体が良い多様性メンバーになります。  「自分自身の出力を特徴量に足す」形ですが、**5-fold CVで作ったOOFなのでリークはしません**。- **なぜ最後の混合を順位空間で行うのか**: 2つのストリームは出力スケールが違います  （dualは1205列、regimeはさらに広い行列から出た確率）。確率のまま重み付き平均すると、  スケールの大きいほうに引っ張られます。`rank01` に直してから混ぜれば、**混合比 0.55 がそのまま意図どおりに効きます**。- **`del` と `gc.collect()` と `torch.cuda.empty_cache()`**: 巨大な中間行列を明示的に解放しています。  Pythonは参照が切れれば自動で解放しますが、GPUメモリはキャッシュに残るため `empty_cache()` が要ります。  数GB級の行列を扱うnotebookでは、**使い終わった変数を消すのも設計の一部**です。> ⚠️ **批判的に見るべき点**: `MIX_W = 0.55`、`FUSION_C = 3.5`、`missing >= 4` という閾値は、> どうやって選ばれたのでしょうか。コードからは分かりません。もしこれらが**Public LBのスコアを見ながら**> 調整されたのなら、Public LBに対する過学習であり、Private LBでは崩れる可能性があります。> notebook中に `VALIDATE = False` というフラグがあり、既定では検証パスを走らせない設計になっている点も、> 「honestなCV評価が表に出ていない」という意味で注意信号です。

In [ ]:
def main():
    print(f"[{datetime.datetime.now():%H:%M:%S}] device={DEVICE}  VALIDATE={VALIDATE}", flush=True)
    oof, test_raw, names, y, train, test = build_pool()
    n, nt = len(y), len(test)
    print(f"[{datetime.datetime.now():%H:%M:%S}] pool built: {names.__len__()} members "
          f"oof={oof.shape} test={test_raw.shape}", flush=True)

    feature_cols = [c for c in train.columns if c not in ("id", "addicted_label")]
    missing = train[feature_cols].isna().sum(axis=1).to_numpy()
    complete = (missing == 0).astype(np.float64)
    missing_many = (missing >= 4).astype(np.float64)
    tm = test[feature_cols].isna().sum(axis=1).to_numpy()
    test_complete = (tm == 0).astype(np.float64)
    test_missing_many = (tm >= 4).astype(np.float64)

    folds = list(StratifiedKFold(N_FOLD, shuffle=True, random_state=SEED).split(np.zeros(n), y))

    # ---- refit meta-stack base member (CVP C=0.1) -- REQUIRED fusion feature ----
    print(f"[{datetime.datetime.now():%H:%M:%S}] refitting meta-stack base (required feature) ...", flush=True)
    base_oof = np.empty(n, dtype=np.float64)
    base_test = np.zeros(nt, dtype=np.float64)
    for fit, valid in folds:
        sc = StandardScaler().fit(oof[fit])
        m = LogisticRegression(C=0.1, max_iter=1200, solver="lbfgs", tol=1e-5)
        m.fit(sc.transform(oof[fit]), y[fit])
        base_oof[valid] = m.predict_proba(sc.transform(oof[valid]))[:, 1]
        base_test += m.predict_proba(sc.transform(test_raw))[:, 1] / len(folds)
    print(f"... base meta-stack pooled OOF = {roc_auc_score(y, base_oof):.6f}", flush=True)
    # the refit meta member also joins the fusion feature matrix (150 cols)
    oof = np.column_stack([oof, base_oof])
    test_raw = np.column_stack([test_raw, base_test])
    print(f"... fusion matrix now {oof.shape[1]} columns", flush=True)

    # ---- full-data fits (these define the test submission) -------------------
    rf = np.column_stack([rank01(oof[:, j]) for j in range(oof.shape[1])])
    rt = np.column_stack([rank01(test_raw[:, j]) for j in range(test_raw.shape[1])])
    bf, bt = np.hstack([rf, logit(oof)]), np.hstack([rt, logit(test_raw)])
    print(f"[{datetime.datetime.now():%H:%M:%S}] fitting dual on full data ...", flush=True)
    dual_test = fit_logreg(bf, y, bt)
    reg_f = regime_feats(rf, bf, complete, missing_many)
    reg_t = regime_feats(rt, bt, test_complete, test_missing_many)
    mean, scale = reg_f.mean(axis=0), reg_f.std(axis=0)
    scale[scale == 0] = 1.0
    reg_f -= mean; reg_f /= scale
    reg_t -= mean; reg_t /= scale
    del rf, bf, rt, bt
    gc.collect(); torch.cuda.empty_cache()
    print(f"[{datetime.datetime.now():%H:%M:%S}] fitting regime on full data ...", flush=True)
    reg_test = fit_logreg(reg_f, y, reg_t)
    del reg_f, reg_t
    gc.collect(); torch.cuda.empty_cache()
    test_mix = rank01(MIX_W * rank01(dual_test) + (1.0 - MIX_W) * rank01(reg_test))

    result = {
        'member_count': len(names),
        'fold_definition': 'StratifiedKFold(5, shuffle=True, random_state=42)',
        'base_meta_oof': float(roc_auc_score(y, base_oof)),
        'VALIDATE': VALIDATE,
        'device': str(DEVICE),
        'generated': datetime.datetime.utcnow().isoformat(),
    }

    if VALIDATE:
        # ---- optional 5-fold OOF (validation) + nested alpha -----------------
        oof_mix = np.empty(n, dtype=np.float64)
        fold_rows = []
        for fold, (fit, valid) in enumerate(folds):
            t0 = time.perf_counter()
            rtr = np.column_stack([rank01(oof[fit, j]) for j in range(oof.shape[1])])
            rva = np.column_stack([rank01(oof[valid, j]) for j in range(oof.shape[1])])
            btr = np.hstack([rtr, logit(oof[fit])])
            bva = np.hstack([rva, logit(oof[valid])])
            dual_pred = fit_logreg(btr, y[fit], bva)
            reg_tr = regime_feats(rtr, btr, complete[fit], missing_many[fit])
            reg_va = regime_feats(rva, bva, complete[valid], missing_many[valid])
            meanv, scalev = reg_tr.mean(axis=0), reg_tr.std(axis=0)
            scalev[scalev == 0] = 1.0
            reg_tr -= meanv; reg_tr /= scalev
            reg_va -= meanv; reg_va /= scalev
            reg_pred = fit_logreg(reg_tr, y[fit], reg_va)
            del rtr, rva, btr, bva, reg_tr, reg_va
            gc.collect(); torch.cuda.empty_cache()
            mix = MIX_W * rank01(dual_pred) + (1.0 - MIX_W) * rank01(reg_pred)
            oof_mix[valid] = rank01(mix)
            fold_rows.append({
                'fold': fold,
                'dual_auc': float(roc_auc_score(y[valid], dual_pred)),
                'regime_auc': float(roc_auc_score(y[valid], reg_pred)),
                'mix_auc': float(roc_auc_score(y[valid], mix)),
                'seconds': round(time.perf_counter() - t0, 2),
            })
            print(json.dumps(fold_rows[-1]), flush=True)
        grid = np.array([0.0, 0.025, 0.05, 0.075, 0.1, 0.15, 0.2, 0.25, 1 / 3, 0.4, 0.5])
        nested = np.empty(n, dtype=np.float64)
        selected = []
        for fold, (fit, valid) in enumerate(folds):
            base_fit = rankdata(base_oof[fit]) / len(fit)
            diri_fit = rankdata(oof_mix[fit]) / len(fit)
            alpha = max(grid, key=lambda a: roc_auc_score(y[fit], (1 - a) * base_fit + a * diri_fit))
            selected.append(float(alpha))
            base_valid = np.searchsorted(np.sort(base_oof[fit]), base_oof[valid], side="right") / len(fit)
            diri_valid = np.searchsorted(np.sort(oof_mix[fit]), oof_mix[valid], side="right") / len(fit)
            nested[valid] = (1 - alpha) * base_valid + alpha * diri_valid
        abar = float(np.mean(selected))
        print("nested selected alpha:", [round(a, 3) for a in selected], "mean", round(abar, 3), flush=True)
        np.save(OUT_DIR / "oof_mix.npy", oof_mix)
        np.save(OUT_DIR / "oof_nested.npy", nested)
        result['pooled_oof_auc_mix'] = float(roc_auc_score(y, oof_mix))
        result['nested_pooled_oof_auc'] = float(roc_auc_score(y, nested))
        result['nested_selected_alpha'] = [round(a, 3) for a in selected]
        result['folds'] = fold_rows
        test_nested = rank01((1 - abar) * base_test + abar * test_mix)
        use_alpha = abar
    else:
        # fast path: no per-fold OOF; nested blend uses the fixed default alpha
        use_alpha = ALPHA_DEFAULT
        test_nested = rank01((1 - use_alpha) * base_test + use_alpha * test_mix)

    # ---- outputs --------------------------------------------------------------
    np.save(OUT_DIR / "test_mix.npy", test_mix)
    np.save(OUT_DIR / "test_nested.npy", test_nested)
    pd.DataFrame({"id": test.id, "addicted_label": test_mix}).to_csv(OUT_DIR / "submission_mix.csv", index=False)
    pd.DataFrame({"id": test.id, "addicted_label": test_nested}).to_csv(OUT_DIR / "submission_nested.csv", index=False)
    print(f"[{datetime.datetime.now():%H:%M:%S}] DONE. wrote submission_mix.csv / submission_nested.csv "
          f"(nested alpha={use_alpha:.3f})", flush=True)
    (OUT_DIR / "result.json").write_text(json.dumps(result, indent=2))
    print(json.dumps(result, indent=2), flush=True)


### 最後のセル: `main()` の呼び出し**何をしているか**: 上で定義した `main()` を実行し、`submission.csv` を生成します。

In [ ]:
main()
